# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

Find and Create an AI model for Nintendo that can successfully play a game of tetris for 1 minute or longer.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

Data was given by Nintendo and was generated through simulation rather than collected from external sources.

In [ ]:
# game.py

class Game:
    def __init__(self, mode, agent=None, generation=None, game_num=None):
        self.board = Board()
        self.curr_piece = Piece()
        self.pieces_dropped = 0
        self.rows_cleared = 0
        self.generation = generation
        self.game_num = game_num
        if mode == "TetriMind":
            if agent is None:
                self.ai = CUSTOM_AI_MODEL()
            else:
                self.ai = agent

    def run_no_visual(self):
        if self.ai is None:
            return -1
        while True:
            x, piece = self.ai.get_best_move(self.board, self.curr_piece)
            self.curr_piece = piece
            y = self.board.drop_height(self.curr_piece, x)
            self.drop(y, x=x)
            if self.board.top_filled():
                break
        return self.pieces_dropped, self.rows_cleared

In [ ]:
# custom_model.py

def evaluate_agent(agent, num_games=5, verbose=False, generation=None):
    from game import Game

    total_rows = 0
    total_pieces = 0
    best_rows = 0

    for game_num in range(num_games):
        game = Game("TetriMind", agent=agent,
                    generation=generation, game_num=game_num + 1)
        pieces_dropped, rows_cleared = game.run_no_visual()

        total_rows += rows_cleared
        total_pieces += pieces_dropped
        best_rows = max(best_rows, rows_cleared)

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

I analyzed the reference implementations to understand which board features correlate with good Tetris play:

Aggregate Height: Total height of all columns (lower is better)

Holes: Empty cells with blocks above them (fewer is better)

Bumpiness: Height variation between adjacent columns (smoother is better)

Lines Cleared: Complete rows (more is better)

Max Height: Tallest column (lower is better to avoid game over)

Wells: Deep gaps between columns

Transitions: Changes from filled to empty cells

Pit Depth: Depth of the deepest well

Blocks Above Holes: Blocks trapped above holes

In [ ]:
# custom_model.py

def extract_features(self, board):
    features = {}

    heights = self.get_column_heights(board)
    features['aggregate_height'] = sum(heights)
    features['max_height'] = max(heights) if heights else 0
    features['lines_cleared'] = self.count_complete_lines(board)
    features['holes'] = self.count_holes(board, heights)
    features['bumpiness'] = self.calculate_bumpiness(heights)
    features['wells'] = self.calculate_wells(heights)
    features['column_transitions'] = self.count_column_transitions(board, heights)
    features['row_transitions'] = self.count_row_transitions(board)
    features['pit_depth'] = self.calculate_pit_depth(heights)
    features['blocks_above_holes'] = self.count_blocks_above_holes(board, heights)

    return features

# 4.Prepare the Data


Apply any data transformations and explain what and why


I transformed the raw 10x20 Tetris board (200 cells) into 10 meaningful features like aggregate height, holes, bumpiness, and lines cleared that capture what makes a good board state. I also edited the game code to make it so multiple lines clear at once when multiple are filled in, rather than one at a time. I normalized the weight ranges to keep the model stable (lines_cleared: 0-2, others: -2 to 0.5) and designed a fitness function that rewards both survival (rows cleared) and efficiency (pieces dropped). All model weights and training history were saved in JSON format to enable multi-session training and recovery.

In [ ]:
# custom_model.py – applying weights to features (feature → score)

def evaluate_move(self, board, piece, x, y):
    board_copy = deepcopy(board.board)
    for pos in piece.body:
        try:
            board_copy[y + pos[1]][x + pos[0]] = True
        except:
            return float('-inf')

    features = self.extract_features(board_copy)
    score = sum(self.weights[key] * features[key] for key in self.weights.keys())
    return score

In [ ]:
# custom_model.py – weight mutation with clamped ranges (data normalization)

def mutate_weights(weights, mutation_rate=0.15, mutation_scale=0.2,
                   mutation_type='gaussian'):
    mutated = weights.copy()
    for key in mutated:
        if random.random() < mutation_rate:
            # different mutation strategies...
            # ...

            # keep weights in reasonable ranges
            if key == 'lines_cleared':
                mutated[key] = max(0.0, min(2.0, mutated[key]))
            else:
                mutated[key] = max(-2.0, min(0.5, mutated[key]))
    return mutated

In [ ]:
# custom_model.py – fitness function (how we score an agent)

avg_rows = total_rows / num_games
avg_pieces = total_pieces / num_games
fitness = avg_rows + (avg_pieces * 0.1)

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


Enhanced Genetic Algorithm is the model I chose, based off the example code.

Its an Advanced genetic algorithm with: Multiple mutation strategies (Gaussian, Uniform, Adaptive), Elite preservation (top 20% kept unchanged), Balanced exploration/exploitation (40% mutations, 40% random), and All-time best tracking (never loses best model becuz it saves in a .json file)

I selected the Enhanced Genetic Algorithm because it has proven Success: Achieved 6.7x improvement over baseline in first generation and ~100x in the most recent generation (12), because it has very thorough training (takes days), and because its Weights are human-readable and explainable.

In [ ]:
# custom_model.py

class CUSTOM_AI_MODEL:
    def __init__(self, weights=None):
        if weights is not None:
            self.weights = weights
        else:
            best_model_data = load_best_model_full()
            if best_model_data:
                self.weights = best_model_data['weights']
                self.generation = best_model_data.get('generation', 0)
                self.rows_cleared = best_model_data.get('rows_cleared', 0)
                print(f"\n{'='*70}")
                print("LOADED BEST MODEL")
                print(f"{'='*70}")
                print(f"Generation: {self.generation}")
                print(f"Best Performance: {self.rows_cleared:,} rows cleared")
                print(f"Weights:")
                for key, value in self.weights.items():
                    print(f"  {key:25s}: {value:8.4f}")
                print(f"{'='*70}\n")
            else:
                self.weights = {#Best weights for 200k model
                    'aggregate_height': 0.5910257582448996,
                    'lines_cleared': 0.5725705820597807,
                    'holes': 0.894026514754244,
                    'bumpiness': 0.18044482983944404,
                    'max_height': 0.3047025227634215,
                    'wells': 0.19975437173237554,
                    'column_transitions': 0.22859171194130873,
                    'row_transitions': 0.19262763539026106,
                    'pit_depth': 0.2650863144881257,
                    'blocks_above_holes': 0.10029466980371468
                }
                self.generation = 0
                self.rows_cleared = 0
                print("Using default weights (no saved model found)")
        
        self.fitness_scores = []
        self.avg_fitness = 0

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


I optimized the genetic algorithm by adding safety features like automatic backups and only saving models that beat the all-time best (not just by best in a generation, leared from heartbreaking experience), and tuned the training to run 20 agents playing 5 games each per generation. This approach improved performance from 2k baseline rows to 200k rows in 12 generations.

In [ ]:
# custom_model.py – improved population composition

if best_weights and current_gen > 1:
    population.append(CUSTOM_AI_MODEL(best_weights))

    for i in range(elite_count - 1):
        mutated = mutate_weights(best_weights, mutation_rate=0.1,
                                 mutation_scale=0.1, mutation_type='gaussian')
        population.append(CUSTOM_AI_MODEL(mutated))

    exploit_count = (population_size - elite_count) // 2

    explore_count = population_size - elite_count - exploit_count


In [ ]:
# custom_model.py – save best model + generation backups

def save_best_model(weights, fitness, generation, rows_cleared):
    filepath = 'best_model.json'
    current_best = load_best_model_full()

    is_new_best = False
    if current_best is None:
        is_new_best = True
    elif rows_cleared > current_best.get('rows_cleared', 0):
        is_new_best = True

    backup_filepath = f'model_gen_{generation}.json'
    backup_data = {
        'weights': weights,
        'fitness': fitness,
        'generation': generation,
        'rows_cleared': rows_cleared,
        'timestamp': datetime.now().isoformat()
    }
    with open(backup_filepath, 'w') as f:
        json.dump(backup_data, f, indent=2)

    if is_new_best:
        with open(filepath, 'w') as f:
            json.dump(backup_data, f, indent=2)

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


I built an AI player called TetriMind that can play Tetris on its own. Its job is to place each piece in the best possible spot so it can survive as long as possible and clear many rows.  

The AI looks at the board and turns it into numbers like “how high is the stack,” “how many holes are there,” and “how smooth is the surface.” It uses learned weights on these features to score every possible move, then picks the move with the best score.  

I used a genetic algorithm, which means many different versions of the AI played lots of games, and the best ones were kept and slightly changed. Over several generations, this process improved the weights so the AI’s decisions got better and better.  

The final AI clearly outperforms the starting models, clearing many more rows and playing much more safely. It is also designed so it can keep training over time, meaning its performance can continue to improve with more generations.

After final training, my model was able to clear 200k rows with the best weights found after 16 generations.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


python3 main.py TetriMind

Ive copied the weights of the best model to the custom_model file, so no need for best_model.json